In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option('display.max_rows', None)


Carregando as bases de dados

In [2]:
virus_core_dataset = pd.read_csv('C:/Users/kened/OneDrive/Desktop/Estudos/Lean/data/raw/virus_core.csv')
virus_core_dataset

,virus_id,name,family,genus,genome_type,strand,enveloped
0,1,SARS-CoV-2,NaN,Betacoronavirus,RNA,ssRNA(+),True
1,2,Influenza A virus,Orthomyxoviridae,NaN,rna,ssRNA(-),True
2,3,HIV-1,Retroviridae,Lentivirus,RNA,ssRNA-RT,True
3,4,Ebola virus,Filoviridae,Ebolavirus,RNA,ssRNA(-),True
4,5,Zika virus,NaN,Flavivirus,RNA,ssRNA(+),True
5,6,Dengue virus,Flaviviridae,NaN,RNA,ssRNA(+),True
6,7,Rabies virus,Rhabdoviridae,Lyssavirus,RNA,ssRNA(-),True
7,8,Hepatitis B virus,Hepadnaviridae,Orthohepadnavirus,DNA,partially dsDNA,True
8,9,Adenovirus,Adenoviridae,Mastadenovirus,DNA,dsDNA,False
9,10,Human papillomavirus,Papillomaviridae,Alphapapillomavirus,DNA,dsDNA,False


In [3]:
virus_epidemiology_dataset = pd.read_csv('C:/Users/kened/OneDrive/Desktop/Estudos/Lean/data/raw/virus_epidemiology.csv')
virus_epidemiology_dataset

,virus_id,transmission,mortality,r0,host
0,1,airborne,2.00,2.50,human
1,2,airborne,0.10,1.30,human
2,3,bodily fluids,80.00,2.00,human
3,4,Airborne,NaN,1.80,human
4,5,vector,0.01,3.00,human
5,6,vector,150.00,4.00,human
6,7,animal bite,99.00,0.50,animal
7,8,airborne,4.10,1.59,NaN
8,9,vector,0.23,500.00,human
9,10,vector,NaN,2.99,animal


In [4]:
virus_target_dataset = pd.read_csv('C:/Users/kened/OneDrive/Desktop/Estudos/Lean/data/raw/virus_target.csv')
virus_target_dataset

,virus_id,risk_category
0,1,low
1,2,low
2,3,high
3,4,medium
4,5,low
5,6,low
6,7,high
7,8,low
8,9,low
9,10,medium


primeiro vamos verificar se há registros duplicados nos datasets

In [5]:
print('shapes:', virus_core_dataset.shape, virus_epidemiology_dataset.shape, virus_target_dataset.shape)
print('registros duplicados em virus_core:', virus_core_dataset['virus_id'].duplicated().sum())
print('registros duplicados em virus_epidemiology:', virus_epidemiology_dataset['virus_id'].duplicated().sum())
print('registros duplicados em virus_target:', virus_target_dataset['virus_id'].duplicated().sum())

shapes: (73, 7) (72, 5) (70, 2)
registros duplicados em virus_core: 3
registros duplicados em virus_epidemiology: 2
registros duplicados em virus_target: 0


Checando melhor os registros para verificar se são exatamente os mesmos

In [6]:
display(virus_core_dataset[virus_core_dataset['virus_id'].duplicated(keep=False)])
display(virus_epidemiology_dataset[virus_epidemiology_dataset['virus_id'].duplicated(keep=False)])

,virus_id,name,family,genus,genome_type,strand,enveloped
2,3,HIV-1,Retroviridae,Lentivirus,RNA,ssRNA-RT,True
15,16,Marburg virus,Filoviridae,Marburgvirus,RNA,ssRNA(-),True
40,41,Escherichia phage ev_rybdy6,Unknown,Unknown,Unknown,Unknown,NaN
70,3,HIV-1,Retroviridae,Lentivirus,RNA,ssRNA-RT,True
71,16,Marburg virus,Filoviridae,Marburgvirus,RNA,ssRNA(-),True
72,41,Escherichia phage ev_rybdy6,Unknown,Unknown,Unknown,Unknown,NaN


,virus_id,transmission,mortality,r0,host
7,8,airborne,4.10,1.59,NaN
33,34,airborne,6.83,2.71,animal
70,8,airborne,4.10,1.59,NaN
71,34,airborne,6.83,2.71,animal


In [7]:
virus_core_dataset = virus_core_dataset.drop_duplicates()
virus_epidemiology_dataset = virus_epidemiology_dataset.drop_duplicates()

print('shapes:', virus_core_dataset.shape, virus_epidemiology_dataset.shape, virus_target_dataset.shape)
print('registros duplicados em virus_core:', virus_core_dataset['virus_id'].duplicated().sum())
print('registros duplicados em virus_epidemiology:', virus_epidemiology_dataset['virus_id'].duplicated().sum())

shapes: (70, 7) (70, 5) (70, 2)
registros duplicados em virus_core: 0
registros duplicados em virus_epidemiology: 0


Identifiquei 3 registros duplicados em virus_core e 2 em virus_epidemiology. Ao inspecionar as linhas, percebi que são duplicatas exatas, sem conflito de valores.
Optei por remover com drop_duplicates() mantendo a primeira ocorrência pois manter duplicados poderia enviesar o modelo dando peso indevido a certos vírus.

In [8]:
dataset_parcial = pd.merge(virus_core_dataset,virus_epidemiology_dataset,on='virus_id',how='inner')
dataset_parcial

,virus_id,name,family,genus,genome_type,strand,enveloped,transmission,mortality,r0,host
0,1,SARS-CoV-2,NaN,Betacoronavirus,RNA,ssRNA(+),True,airborne,2.00,2.50,human
1,2,Influenza A virus,Orthomyxoviridae,NaN,rna,ssRNA(-),True,airborne,0.10,1.30,human
2,3,HIV-1,Retroviridae,Lentivirus,RNA,ssRNA-RT,True,bodily fluids,80.00,2.00,human
3,4,Ebola virus,Filoviridae,Ebolavirus,RNA,ssRNA(-),True,Airborne,NaN,1.80,human
4,5,Zika virus,NaN,Flavivirus,RNA,ssRNA(+),True,vector,0.01,3.00,human
5,6,Dengue virus,Flaviviridae,NaN,RNA,ssRNA(+),True,vector,150.00,4.00,human
6,7,Rabies virus,Rhabdoviridae,Lyssavirus,RNA,ssRNA(-),True,animal bite,99.00,0.50,animal
7,8,Hepatitis B virus,Hepadnaviridae,Orthohepadnavirus,DNA,partially dsDNA,True,airborne,4.10,1.59,NaN
8,9,Adenovirus,Adenoviridae,Mastadenovirus,DNA,dsDNA,False,vector,0.23,500.00,human
9,10,Human papillomavirus,Papillomaviridae,Alphapapillomavirus,DNA,dsDNA,False,vector,NaN,2.99,animal


In [9]:
dataset_full = pd.merge(dataset_parcial,virus_target_dataset,on='virus_id',how='inner')
dataset_full

,virus_id,name,family,genus,genome_type,strand,enveloped,transmission,mortality,r0,host,risk_category
0,1,SARS-CoV-2,NaN,Betacoronavirus,RNA,ssRNA(+),True,airborne,2.00,2.50,human,low
1,2,Influenza A virus,Orthomyxoviridae,NaN,rna,ssRNA(-),True,airborne,0.10,1.30,human,low
2,3,HIV-1,Retroviridae,Lentivirus,RNA,ssRNA-RT,True,bodily fluids,80.00,2.00,human,high
3,4,Ebola virus,Filoviridae,Ebolavirus,RNA,ssRNA(-),True,Airborne,NaN,1.80,human,medium
4,5,Zika virus,NaN,Flavivirus,RNA,ssRNA(+),True,vector,0.01,3.00,human,low
5,6,Dengue virus,Flaviviridae,NaN,RNA,ssRNA(+),True,vector,150.00,4.00,human,low
6,7,Rabies virus,Rhabdoviridae,Lyssavirus,RNA,ssRNA(-),True,animal bite,99.00,0.50,animal,high
7,8,Hepatitis B virus,Hepadnaviridae,Orthohepadnavirus,DNA,partially dsDNA,True,airborne,4.10,1.59,NaN,low
8,9,Adenovirus,Adenoviridae,Mastadenovirus,DNA,dsDNA,False,vector,0.23,500.00,human,low
9,10,Human papillomavirus,Papillomaviridae,Alphapapillomavirus,DNA,dsDNA,False,vector,NaN,2.99,animal,medium


Juntei os datasets pela chave virus_id para ter o panorâma geral dos dados em um só dataframe, agora o próximo passo é verificar 
valores ausentes,em quais colunas,estatística descritiva,Cardinalidade,tipos de dados etc

In [10]:
dataset_full.describe()

,virus_id,mortality,r0
count,70.000000,67.000000,70.000000
mean,35.500000,11.076716,9.105857
std,20.351085,25.440637,59.531158
min,1.000000,-5.000000,-1.200000
25%,18.250000,2.360000,1.305000
50%,35.500000,5.910000,2.025000
75%,52.750000,8.155000,2.805000
max,70.000000,150.000000,500.000000


In [11]:
dataset_full.isnull().sum()

virus_id          0
name              0
family            7
genus             4
genome_type       0
strand            0
enveloped        41
transmission      0
mortality         3
r0                0
host              6
risk_category     0
dtype: int64

In [12]:
dataset_full.dtypes

virus_id           int64
name              object
family            object
genus             object
genome_type       object
strand            object
enveloped         object
transmission      object
mortality        float64
r0               float64
host              object
risk_category     object
dtype: object

In [13]:
dataset_full.nunique()

virus_id         70
name             70
family           18
genus            20
genome_type       6
strand            8
enveloped         2
transmission      8
mortality        66
r0               65
host              3
risk_category     3
dtype: int64

In [20]:
for var in ['genome_type', 'strand', 'enveloped', 'transmission']:
    print(dataset_full[var].value_counts(dropna=False))
    print()

genome_type
Unknown    41
RNA        19
DNA         7
rna         1
RNA         1
Rna         1
Name: count, dtype: int64

strand
Unknown            41
ssRNA(+)           10
ssRNA(-)            8
dsDNA               7
partially dsDNA     1
ssRNA-RT            1
dsRNA               1
ssRNA               1
Name: count, dtype: int64

enveloped
NaN      41
True     25
False     4
Name: count, dtype: int64

transmission
vector           22
bodily fluids    15
contact          15
airborne         14
Airborne          1
animal bite       1
vectorr           1
air borne         1
Name: count, dtype: int64



In [21]:
for var in ['host', 'family', 'genus']:
    print(dataset_full[var].value_counts(dropna=False))
    print()

host
both      25
animal    20
human     19
NaN        6
Name: count, dtype: int64

family
Unknown             39
NaN                  7
Flaviviridae         3
Herpesviridae        3
Poxviridae           2
Coronaviridae        2
Filoviridae          2
Retroviridae         1
Orthomyxoviridae     1
Hepadnaviridae       1
Rhabdoviridae        1
Caliciviridae        1
Papillomaviridae     1
Adenoviridae         1
Reoviridae           1
Paramyxoviridae      1
Hantaviridae         1
Arenaviridae         1
Pneumoviridae        1
Name: count, dtype: int64

genus
Unknown                41
NaN                     4
Betacoronavirus         3
Flavivirus              3
Orthopoxvirus           2
Simplexvirus            2
Lentivirus              1
Ebolavirus              1
Lyssavirus              1
Alphacoronavirus        1
Orthohepadnavirus       1
Mastadenovirus          1
Alphapapillomavirus     1
Marburgvirus            1
Rotavirus               1
Norovirus               1
Mammarenavirus         

In [22]:
print(dataset_full['transmission'].value_counts(dropna=False))

transmission
vector           22
bodily fluids    15
contact          15
airborne         14
Airborne          1
animal bite       1
vectorr           1
air borne         1
Name: count, dtype: int64


In [23]:
print(dataset_full['genus'].value_counts(dropna=False))

genus
Unknown                41
NaN                     4
Betacoronavirus         3
Flavivirus              3
Orthopoxvirus           2
Simplexvirus            2
Lentivirus              1
Ebolavirus              1
Lyssavirus              1
Alphacoronavirus        1
Orthohepadnavirus       1
Mastadenovirus          1
Alphapapillomavirus     1
Marburgvirus            1
Rotavirus               1
Norovirus               1
Mammarenavirus          1
Orthopneumovirus        1
Orthorubulavirus        1
Morbillivirus           1
Varicellovirus          1
Name: count, dtype: int64


In [24]:
unknowns = dataset_full['genome_type'] == 'Unknown'
dataset_full.loc[unknowns, ['genome_type', 'strand', 'genus', 'enveloped', 'family']]

,genome_type,strand,genus,enveloped,family
29,Unknown,Unknown,Unknown,NaN,Unknown
30,Unknown,Unknown,Unknown,NaN,Unknown
31,Unknown,Unknown,Unknown,NaN,Unknown
32,Unknown,Unknown,Unknown,NaN,Unknown
33,Unknown,Unknown,Unknown,NaN,Unknown
34,Unknown,Unknown,Unknown,NaN,Unknown
35,Unknown,Unknown,Unknown,NaN,Unknown
36,Unknown,Unknown,Unknown,NaN,Unknown
37,Unknown,Unknown,Unknown,NaN,Unknown
38,Unknown,Unknown,Unknown,NaN,Unknown


O dataset_full tem 70 linhas e 12 colunas. Rodei describe, isnull, dtypes e nunique e encontrei os seguintes pontos:
- Nulos: enveloped (41), family (7), host (6), genus (4), mortality (3). O target não tem nulos.
- Valores improváveis: mortality vai de -5 a 150. Como o dicionário define a variável como percentual de óbitos, a faixa válida que irei considerar é de 0 a 100.
- Tipos: o dicionário diz que enveloped deveria ser booleano (True/False), mas está como object por causa dos nulos.
- Padronização: genome_type tem 6 valores únicos, mas o dicionário diz que deve ter apenas RNA ou DNA. Podemos ver que existem valores escritos de formas diferentes, precisamos padronizar. O mesmo acontece em transmission, que tem typos como "Airborne", "air borne" e "vectorr".
- Mistura de Unknown e NaN: family e genus têm tanto "Unknown" quanto NaN convivendo, são duas formas diferentes de representar a mesma coisa e precisam ser unificadas.
- O valor "Unknown" aparece em 41 linhas em genome_type, strand e genus, e enveloped tem 41 NaN. Confirmei que são as mesmas 41 linhas, ou seja, um bloco de vírus sem caracterização biológica. Vou manter esses registros e tratar "Unknown" como categoria válida, descartar significaria perder 58% da base e imputar inventaria informação.
- Nomes: name tem 70 valores únicos em 70 linhas, assim como o ID não há como enriquecer o modelo com essa feature portanto será removida